In [ ]:
import asyncio
import aiohttp
import time
import os
import json
import hashlib
import logging
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
from collections import defaultdict
from urllib.parse import urlparse
from bs4 import BeautifulSoup
from firecrawl import FirecrawlApp
from google.oauth2 import service_account
from googleapiclient.discovery import build

# Constants from original pipeline
CATEGORY_THRESHOLDS = {
	"ABOUT_US": 200,
	"EBOOK": 200,
	"COURSES": 300,
	"RECENT_BLOG": 450,
	"TESTIMONIALS": 100,
	"WEBINAR": 150,
	"SERVICES": 150,
	"PODCAST": 200,
	"SHOP": 100,
}

CATEGORY_KEYWORDS = {
	"ABOUT_US": [
		"about", "who-we-are", "company", "our-story", "mission", "values", 
		"about-us", "story", "timeline", "milestones", "why-us"
	],
	
	"EBOOK": [
		"ebook", "e-book", "whitepaper", "white-paper", "guide", "pdf", 
		"resources", "downloads", "books", "library", "documents"
	],
	
	"COURSES": [
		"course", "academy", "learning", "training", "workshop",
		"certification", "program", "bootcamp", "masterclass", 
		"education", "class", "e-learning"
	],
	
	"RECENT_BLOG": [
		"blog", "insights", "articles", "news", "updates", 
		"post", "media", "latest", "trends", "press", 
		"content-hub"
	],
	
	"TESTIMONIALS": [
		"testimonial", "reviews", "case-study", "success-story", 
		"client-story", "customer-story", "feedback", "clients", 
		"portfolio", "results", "social-proof"
	],
	
	"WEBINAR": [
		"webinar", "event", "session", "live", "virtual-event", 
		"presentation", "conference", "summit", "registration", 
		"upcoming", "schedule"
	],
	
	"SERVICES": [
		"service", "solution", "offering", "expertise", "consulting", 
		"what-we-do", "capability", "support", "practice", 
		"professional-services", "how-we-help"
	],
	
	"PODCAST": [
		"podcast", "episodes", "audio", "listen", "show", 
		"interview", "series", "stream", "speakers", 
		"voice", "subscribe"
	],
	
	"SHOP": [
		"shop", "store", "buy", "purchase", "products", 
		"cart", "checkout", "pricing", "e-commerce", 
		"merchandise", "order"
	]
}

COLUMN_TO_WRITE_URL_TO = {
	"ABOUT_US": "M",
	"EBOOK": "N",
	"COURSES": "O",
	"RECENT_BLOG": "P",
	"TESTIMONIALS": "Q",
	"WEBINAR": "R",
	"SERVICES": "S",
	"PODCAST": "T",
	"SHOP": "U"
}

CATEGORY_RULES = {
	'ABOUT_US': 'ascending',
	'EBOOK': 'ascending',
	'COURSES': 'ascending',
	'RECENT_BLOG': 'descending',
	'TESTIMONIALS': 'ascending',
	'WEBINAR': 'descending',
	'SERVICES': 'descending',
	'PODCAST': 'descending',
	'SHOP': 'ascending'
}

EXTRACTION_METADATA_COLUMN = "V"
CACHE_DIR = "firecrawl_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

@dataclass
class ProcessingResult:
	category: str
	content: str
	metadata: str
	urls_found: List[str]

@dataclass
class ProductionConfig:
	# Concurrency limits
	max_concurrent_rows: int = 5
	max_concurrent_scrapes_per_row: int = 3
	max_connections_per_host: int = 10
	
	# Rate limiting
	firecrawl_rate_limit_seconds: float = 8.0
	google_sheets_batch_size: int = 10
	google_sheets_rate_limit: float = 1.0
	
	# Timeouts
	scraping_timeout_seconds: int = 15
	firecrawl_timeout_seconds: int = 30
	worker_timeout_seconds: int = 300
	
	# Retry settings
	max_retries: int = 2
	retry_delay_seconds: float = 1.0
	
	# Memory management
	max_content_size_bytes: int = 50000
	max_cache_size: int = 1000

class ProductionMonitor:
	def __init__(self):
		self.stats = defaultdict(int)
		self.start_time = time.time()
		self.errors = []
		
		# Setup logging
		logging.basicConfig(
			level=logging.INFO,
			format='%(asctime)s - %(levelname)s - %(message)s',
			handlers=[
				logging.FileHandler('pipeline.log'),
				logging.StreamHandler()
			]
		)
		self.logger = logging.getLogger(__name__)

	def log_progress(self, processed: int, total: int):
		elapsed = time.time() - self.start_time
		rate = processed / elapsed if elapsed > 0 else 0
		eta = (total - processed) / rate if rate > 0 else 0
		
		self.logger.info(f"Progress: {processed}/{total} ({processed/total*100:.1f}%) "
						f"Rate: {rate:.2f} rows/sec, ETA: {eta/60:.1f} min")

	def log_error(self, error: str, row_num: Optional[int] = None):
		error_msg = f"Row {row_num}: {error}" if row_num else error
		self.errors.append(error_msg)
		self.logger.error(error_msg)

	def log_stats(self):
		self.logger.info("Pipeline Statistics:")
		for key, value in self.stats.items():
			self.logger.info(f"  {key}: {value}")
		
		if self.errors:
			self.logger.error(f"Errors encountered: {len(self.errors)}")
			for error in self.errors[-10:]:
				self.logger.error(f"  {error}")

# Helper functions from original pipeline
def calculate_url_depth(url: str) -> int:
	try:
		parsed = urlparse(url)
		path = parsed.path.strip('/').split('/')
		return len(path)
	except Exception:
		return -1

def truncate_to_bytes(text: str, max_bytes: int) -> str:
	encoded = text.encode('utf-8')
	if len(encoded) <= max_bytes:
		return text
	truncated = encoded[:max_bytes]
	return truncated.decode('utf-8', errors='ignore')

def safe_join(contents: List[str], max_bytes: int = 50000, delimiter: str = " --- NEXT CONTENT FROM HERE --- ") -> str:
	final_text = ""
	for content in contents:
		candidate = final_text + (delimiter if final_text else "") + content
		if len(candidate.encode('utf-8')) > max_bytes:
			break
		final_text = candidate
	return final_text

def extract_main_html_content(html: str) -> str:
	soup = BeautifulSoup(html, "html.parser")
	for tag in soup(["script", "style", "noscript"]):
		tag.decompose()
	main = soup.find("main") or soup.find("article")
	if main:
		return main.get_text(separator="\n", strip=True)
	candidates = [
		div for div in soup.find_all("div")
		if len(div.get_text(strip=True)) > 200
		   and not any(c in " ".join(div.get("class", [])).lower() for c in ["nav", "header", "footer", "popup"])
	]
	if candidates:
		return max(candidates, key=lambda d: len(d.get_text(strip=True))).get_text(separator="\n", strip=True)
	return soup.get_text(separator="\n", strip=True)

class OptimizedFirecrawlWrapper:
	def __init__(self, api_key):
		self.app = FirecrawlApp(api_key=api_key)
		self.cache = {}
		
	def _hash_url(self, url: str) -> str:
		return hashlib.md5(url.encode()).hexdigest()

	def _get_cache_path(self, url: str) -> str:
		return os.path.join(CACHE_DIR, f"{self._hash_url(url)}.json")
	
	def get_cached_result(self, url: str) -> Optional[List[str]]:
		"""Check if URL is cached"""
		cache_path = self._get_cache_path(url)
		if os.path.exists(cache_path):
			try:
				with open(cache_path, 'r') as f:
					links = json.load(f)
					return links
			except Exception:
				return None
		return None

	def map_url(self, url: str) -> List[str]:
		"""Map URL with caching"""
		cached = self.get_cached_result(url)
		if cached:
			return cached
			
		try:
			result = self.app.map_url(url)
			if getattr(result, 'success', False):
				links = result.links
				# Cache the result
				cache_path = self._get_cache_path(url)
				with open(cache_path, 'w') as f:
					json.dump(links, f, indent=2)
				return links
			else:
				return []
		except Exception as e:
			print(f"Firecrawl error for {url}: {e}")
			return []

	def filter_by_category(self, urls: List[str], category: str) -> List[str]:
		keywords = CATEGORY_KEYWORDS.get(category.upper(), [])
		if not keywords:
			return []
		return [u for u in urls if any(k in u.lower() for k in keywords)]

class GoogleSheetsManager:
	def __init__(self, credentials_file: str):
		scopes = ['https://www.googleapis.com/auth/spreadsheets']
		creds = service_account.Credentials.from_service_account_file(credentials_file, scopes=scopes)
		self.service = build('sheets', 'v4', credentials=creds)

	def extract_spreadsheet_id(self, sheet_url: str) -> str:
		import re
		pattern = r'/spreadsheets/d/([a-zA-Z0-9-_]+)'
		match = re.search(pattern, sheet_url)
		if match:
			return match.group(1)
		raise ValueError("Invalid Google Sheet URL")

	def get_urls(self, spreadsheet_id: str, start_row: int = 2) -> List[Tuple[int, str]]:
		range_name = f"G{start_row}:G"  # Column G for URLs
		result = self.service.spreadsheets().values().get(spreadsheetId=spreadsheet_id, range=range_name).execute()
		values = result.get('values', [])
		return [(i + start_row, row[0]) for i, row in enumerate(values) if row and row[0].strip()]

	def batch_update_cells(self, spreadsheet_id: str, updates: List[Dict]):
		"""Batch update multiple cells at once"""
		if not updates:
			return
			
		body = {
			'valueInputOption': 'RAW',
			'data': updates
		}
		
		self.service.spreadsheets().values().batchUpdate(
			spreadsheetId=spreadsheet_id, 
			body=body
		).execute()

class OptimizedPipeline:
	def __init__(self, config: ProductionConfig, firecrawl_api_key: str, credentials_file: str):
		self.config = config
		self.firecrawl = OptimizedFirecrawlWrapper(firecrawl_api_key)
		self.sheet_mgr = GoogleSheetsManager(credentials_file)
		self.monitor = ProductionMonitor()
		
		# Connection pooling
		self.connector = aiohttp.TCPConnector(
			limit=100,
			limit_per_host=config.max_connections_per_host,
			keepalive_timeout=30,
			enable_cleanup_closed=True
		)
		self.session = None
		
		# Rate limiting
		self.firecrawl_semaphore = asyncio.Semaphore(1)
		self.last_firecrawl_call = 0
		
		# Work queues
		self.task_queue = asyncio.Queue()
		self.results_queue = asyncio.Queue()
		
		# Circuit breaker
		self.firecrawl_failures = 0
		self.max_firecrawl_failures = 10

	async def __aenter__(self):
		self.session = aiohttp.ClientSession(connector=self.connector)
		return self

	async def __aexit__(self, exc_type, exc_val, exc_tb):
		if self.session:
			await self.session.close()
		await self.connector.close()

	async def smart_firecrawl_call(self, url: str) -> List[str]:
		"""Firecrawl call with intelligent rate limiting and 429 error handling"""
		cached_result = self.firecrawl.get_cached_result(url)
		if cached_result:
			self.monitor.logger.info(f"Cache hit for {url}")
			return cached_result
			
		# Circuit breaker
		if self.firecrawl_failures >= self.max_firecrawl_failures:
			self.monitor.log_error("Firecrawl circuit breaker open")
			return []
			
		# Only apply rate limiting for actual API calls
		async with self.firecrawl_semaphore:
			current_time = time.time()
			time_since_last_call = current_time - self.last_firecrawl_call
			
			if time_since_last_call < self.config.firecrawl_rate_limit_seconds:
				wait_time = self.config.firecrawl_rate_limit_seconds - time_since_last_call
				self.monitor.logger.info(f"Rate limiting: waiting {wait_time:.1f}s for {url}")
				await asyncio.sleep(wait_time)
			
			# Try with retry logic for 429 errors
			for attempt in range(3):  # Allow up to 3 retries for 429 errors
				try:
					result = self.firecrawl.map_url(url)
					self.last_firecrawl_call = time.time()
					self.firecrawl_failures = 0  # Reset on success
					self.monitor.stats['firecrawl_success'] += 1
					return result
				except Exception as e:
					error_str = str(e)
					
					# Check for rate limit error (429)
					if "Status code 429" in error_str and "please retry after" in error_str:
						# Extract the retry time from the error message
						import re
						retry_match = re.search(r'please retry after (\d+)s', error_str)
						
						if retry_match:
							retry_seconds = int(retry_match.group(1)) + 1  # Add 1 extra second as buffer
							self.monitor.logger.info(f"Rate limit exceeded. Waiting {retry_seconds}s before retrying {url}")
							await asyncio.sleep(retry_seconds)
							continue  # Try again after waiting
					
					# For other errors or if we can't parse the retry time
					self.firecrawl_failures += 1
					self.monitor.log_error(f"Firecrawl error for {url}: {e}")
					self.monitor.stats['firecrawl_failures'] += 1
					return []

	async def scrape_url_with_session(self, url: str) -> str:
		"""Scrape single URL using shared session"""
		try:
			async with self.session.get(url, timeout=aiohttp.ClientTimeout(total=self.config.scraping_timeout_seconds)) as response:
				if response.status == 200:
					html = await response.text()
					return extract_main_html_content(html)
				else:
					return f"Error: HTTP {response.status}"
		except Exception as e:
			return f"Error: {str(e)}"

	async def scrape_multiple_urls(self, urls: List[str]) -> List[str]:
		"""Scrape multiple URLs concurrently with retry logic"""
		if not urls:
			return []
			
		semaphore = asyncio.Semaphore(self.config.max_concurrent_scrapes_per_row)
		
		async def scrape_with_retry(url: str) -> str:
			async with semaphore:
				for attempt in range(self.config.max_retries):
					try:
						result = await self.scrape_url_with_session(url)
						if not result.startswith("Error:"):
							self.monitor.stats['scraping_success'] += 1
							return result
					except Exception as e:
						if attempt == self.config.max_retries - 1:
							self.monitor.stats['scraping_failures'] += 1
							return f"Error: {str(e)}"
						await asyncio.sleep(self.config.retry_delay_seconds)
				return f"Error: Max retries exceeded"
		
		tasks = [scrape_with_retry(url) for url in urls]
		results = await asyncio.gather(*tasks, return_exceptions=True)
		
		# Convert exceptions to error strings
		return [str(r) if isinstance(r, Exception) else r for r in results]

	async def process_single_category(self, sub_urls: List[str], category: str) -> ProcessingResult:
		"""Process one category for a row"""
		filtered_urls = self.firecrawl.filter_by_category(sub_urls, category)
		
		if not filtered_urls:
			return ProcessingResult(
				category=category,
				content="No URL found",
				metadata=f"{category.upper()}=0",
				urls_found=[]
			)

		# URL depth ranking and selection
		url_depth_pairs = [(url, calculate_url_depth(url)) for url in filtered_urls]
		url_depth_pairs = [(url, depth) for url, depth in url_depth_pairs if depth != -1]
		
		if not url_depth_pairs:
			return ProcessingResult(
				category=category,
				content="No URL found",
				metadata=f"{category.upper()}=0",
				urls_found=[]
			)

		# Apply category-specific sorting rules
		sort_order = CATEGORY_RULES.get(category.upper(), 'ascending')
		reverse_sort = sort_order == 'descending'
		sorted_pairs = sorted(url_depth_pairs, key=lambda x: x[1], reverse=reverse_sort)
		
		# Select top URLs
		selected_urls = [url for url, _ in sorted_pairs[:10]]
		
		# Scrape all selected URLs concurrently
		contents = await self.scrape_multiple_urls(selected_urls)
		
		# Filter out failed scrapes
		valid_contents = [c for c in contents if c and not c.startswith("Error:")]
		
		if not valid_contents:
			content = "No meaningful content found"
			metadata = f"{category.upper()}=0"
		else:
			content = safe_join(valid_contents, self.config.max_content_size_bytes)
			metadata = f"{category.upper()}={len(valid_contents)}"

		return ProcessingResult(
			category=category,
			content=content,
			metadata=metadata,
			urls_found=selected_urls
		)

	async def process_all_categories_for_row(self, row_num: int, main_url: str) -> Dict[str, ProcessingResult]:
		"""Process ALL 9 categories for a single row simultaneously"""
		self.monitor.logger.info(f"Processing all categories for row {row_num}: {main_url}")
		
		# Single Firecrawl call for all categories
		sub_urls = await self.smart_firecrawl_call(main_url)
		
		if not sub_urls:
			# Return empty results for all categories
			empty_results = {}
			for category in CATEGORY_KEYWORDS.keys():
				empty_results[category] = ProcessingResult(
					category=category,
					content="No URL found",
					metadata=f"{category}=0",
					urls_found=[]
				)
			return empty_results
		
		# Process all categories concurrently
		tasks = []
		for category in CATEGORY_KEYWORDS.keys():
			task = self.process_single_category(sub_urls, category)
			tasks.append((category, task))
		
		# Wait for all categories to complete
		results = {}
		for category, task in tasks:
			try:
				result = await task
				results[category] = result
			except Exception as e:
				self.monitor.log_error(f"Error processing category {category} for row {row_num}: {e}", row_num)
				results[category] = ProcessingResult(
					category=category,
					content=f"Error: {str(e)}",
					metadata=f"{category}=0",
					urls_found=[]
				)
		
		return results

	async def worker(self, worker_id: int):
		"""Queue-based worker for processing rows"""
		processed_count = 0
		
		while True:
			try:
				row_data = await asyncio.wait_for(
					self.task_queue.get(),
					timeout=self.config.worker_timeout_seconds
				)
				
				if row_data is None:  # Shutdown signal
					break
				
				row_num, main_url = row_data
				start_time = time.time()
				
				try:
					# Process all categories for this row
					results = await self.process_all_categories_for_row(row_num, main_url)
					processing_time = time.time() - start_time
					
					# Put results in results queue
					await self.results_queue.put((row_num, results))
					processed_count += 1
					
					self.monitor.stats['rows_processed'] += 1
					self.monitor.stats['total_processing_time'] += processing_time
					
					if processed_count % 5 == 0:
						self.monitor.logger.info(f"Worker {worker_id} processed {processed_count} rows")
				
				except Exception as e:
					self.monitor.log_error(f"Worker {worker_id} failed on row {row_num}: {e}", row_num)
					self.monitor.stats['worker_failures'] += 1
				
				finally:
					self.task_queue.task_done()
					
			except asyncio.TimeoutError:
				self.monitor.log_error(f"Worker {worker_id} timeout")
				break
			except Exception as e:
				self.monitor.log_error(f"Worker {worker_id} unexpected error: {e}")
				break

	async def batch_update_sheets(self, batch_results: List[Tuple[int, Dict[str, ProcessingResult]]]):
		"""Batch update Google Sheets to reduce API calls"""
		if not batch_results:
			return
		
		# Prepare batch updates
		updates = []
		
		for row_num, results in batch_results:
			# Update content columns
			for category, result in results.items():
				column = COLUMN_TO_WRITE_URL_TO.get(category)
				if column:
					updates.append({
						'range': f"{column}{row_num}",
						'values': [[result.content]]
					})
			
			# Update metadata column - combine all metadata
			all_metadata = [result.metadata for result in results.values()]
			combined_metadata = ','.join(all_metadata)
			updates.append({
				'range': f"{EXTRACTION_METADATA_COLUMN}{row_num}",
				'values': [[combined_metadata]]
			})
		
		# Perform batch update
		try:
			self.sheet_mgr.batch_update_cells(self.spreadsheet_id, updates)
			self.monitor.stats['sheets_updates'] += len(batch_results)
		except Exception as e:
			self.monitor.log_error(f"Batch sheets update failed: {e}")

	async def process_results(self, total_rows: int):
		"""Process results as they come in and batch update sheets"""
		processed = 0
		batch_results = []
		
		while processed < total_rows:
			try:
				row_num, results = await asyncio.wait_for(
					self.results_queue.get(),
					timeout=self.config.worker_timeout_seconds
				)
				
				batch_results.append((row_num, results))
				processed += 1
				
				# Progress reporting
				if processed % 5 == 0 or processed == total_rows:
					self.monitor.log_progress(processed, total_rows)
				
				# BATCH WRITE TRIGGER: Write to sheets when batch is full OR all rows processed
				if len(batch_results) >= self.config.google_sheets_batch_size or processed == total_rows:
					self.monitor.logger.info(f"📝 WRITING BATCH: {len(batch_results)} rows to Google Sheets")
					await self.batch_update_sheets(batch_results)
					self.monitor.logger.info(f"✅ BATCH WRITTEN: Rows {[r[0] for r in batch_results]} updated in sheets")
					batch_results = []
					
					# Rate limiting for Google Sheets API
					await asyncio.sleep(self.config.google_sheets_rate_limit)
					
			except asyncio.TimeoutError:
				self.monitor.log_error("Timeout waiting for results")
				break

	async def process_all_optimized(self, sheet_url: str, start_row: int = 2):
		"""Main optimized processing function"""
		try:
			self.monitor.logger.info("Starting optimized pipeline")
			self.spreadsheet_id = self.sheet_mgr.extract_spreadsheet_id(sheet_url)
			urls = self.sheet_mgr.get_urls(self.spreadsheet_id, start_row)
			
			self.monitor.logger.info(f"Processing {len(urls)} rows with {self.config.max_concurrent_rows} workers")
			
			# Add all rows to task queue
			for row_num, main_url in urls:
				await self.task_queue.put((row_num, main_url))
			
			# Start workers
			workers = []
			for i in range(self.config.max_concurrent_rows):
				worker_task = asyncio.create_task(self.worker(i))
				workers.append(worker_task)
			
			# Start results processor
			results_processor = asyncio.create_task(self.process_results(len(urls)))
			
			# Wait for all tasks to complete
			await self.task_queue.join()
			
			# Shutdown workers
			for _ in workers:
				await self.task_queue.put(None)
			
			await asyncio.gather(*workers, return_exceptions=True)
			await results_processor
			
			self.monitor.log_stats()
			self.monitor.logger.info("Pipeline completed successfully")
			
		except Exception as e:
			self.monitor.log_error(f"Pipeline failed: {e}")
			raise

In [2]:
FIRECRAWL_API="fc-93c5c61653de4896baf844f3620e036e"
CREDENTIALS_FILE = 'data/url-to-email-445616-cebe4868914f.json'
GOOGLE_SHEET_URL = "https://docs.google.com/spreadsheets/d/1FFZ0c_SgpAcYgLU2gtVx4IFCFhOcQqf8DRdsarhCd_I/edit?gid=0#gid=0" 

In [3]:
# Usage example
async def main():
	config = ProductionConfig(
		max_concurrent_rows=5,
		max_concurrent_scrapes_per_row=3,
		firecrawl_rate_limit_seconds=10.0,
		max_retries=2
	)
	async with OptimizedPipeline(
		config=config,
		firecrawl_api_key=FIRECRAWL_API,
		credentials_file=CREDENTIALS_FILE
	) as pipeline:
		await pipeline.process_all_optimized(
			sheet_url=GOOGLE_SHEET_URL,
			start_row=2
		)

In [4]:
await main()

2025-09-09 01:55:32,892 - INFO - Starting optimized pipeline
2025-09-09 01:55:33,972 - INFO - Processing 352 rows with 5 workers
2025-09-09 01:55:33,973 - INFO - Processing all categories for row 2: https://blackdiamond.net
2025-09-09 01:55:33,974 - INFO - Cache hit for https://blackdiamond.net
2025-09-09 01:55:33,974 - INFO - Processing all categories for row 3: https://1903solutions.com
2025-09-09 01:55:40,075 - INFO - Processing all categories for row 4: https://livewellbehavioralhealth.com
2025-09-09 01:55:40,076 - INFO - Cache hit for https://livewellbehavioralhealth.com
2025-09-09 01:55:40,076 - INFO - Processing all categories for row 5: https://evolutiondevelopmentgroup.com
2025-09-09 01:55:40,077 - INFO - Rate limiting: waiting 10.0s for https://evolutiondevelopmentgroup.com
2025-09-09 01:55:40,077 - INFO - Processing all categories for row 6: https://4oasllc.com
2025-09-09 01:55:40,078 - INFO - Cache hit for https://4oasllc.com
2025-09-09 01:55:41,336 - INFO - Processing all 

Firecrawl error for https://gravystack.com: Unexpected error during map: Status code 429. Rate limit exceeded. Consumed (req/min): 6, Remaining (req/min): 0. Upgrade your plan at https://firecrawl.dev/pricing for increased rate limits or please retry after 1s, resets at Mon Sep 08 2025 20:57:28 GMT+0000 (Coordinated Universal Time) - No additional error details provided.


2025-09-09 02:27:39,846 - INFO - Rate limiting: waiting 10.0s for https://aiconsensus.org
2025-09-09 02:27:51,412 - INFO - Rate limiting: waiting 10.0s for https://scaletime.co
2025-09-09 02:27:51,727 - INFO - Worker 4 processed 45 rows
2025-09-09 02:27:51,728 - INFO - Processing all categories for row 246: https://eightbitstudios.com
2025-09-09 02:27:51,729 - INFO - Progress: 240/352 (68.2%) Rate: 0.12 rows/sec, ETA: 15.1 min
2025-09-09 02:27:51,730 - INFO - 📝 WRITING BATCH: 10 rows to Google Sheets
2025-09-09 02:27:54,119 - INFO - ✅ BATCH WRITTEN: Rows [224, 236, 229, 239, 235, 234, 237, 238, 240, 242] updated in sheets
2025-09-09 02:27:56,328 - INFO - Processing all categories for row 247: https://writemybooks.com
2025-09-09 02:28:02,986 - INFO - Rate limiting: waiting 10.0s for https://awtana.com
2025-09-09 02:28:07,330 - INFO - Processing all categories for row 248: https://newbeginningsconsultation.com
2025-09-09 02:28:07,332 - INFO - Cache hit for https://newbeginningsconsultati